<a href="https://colab.research.google.com/github/SentineNet-AI-SpringBoard/SentinelNet-AI/blob/SENTINELNET-AI-KUSHAGRA/milestone3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

file_path = '/content/drive/MyDrive/collab sentinl net ai/Wednesday-workingHours.pcap_ISCX.csv'
df = pd.read_csv(file_path)

display(df.head())

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


on 10% of total dataset

In [11]:
df_sampled = df.sample(frac=0.1, random_state=42)
display(df_sampled.head())

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
15364,53,105939,1,1,41,131,41,41,41.00,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
18606,443,1319967,10,10,851,3492,267,0,85.10,99.870416,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
454868,53,23894,1,1,64,117,64,64,64.00,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
123291,80,83635124,8,6,366,11595,366,0,45.75,129.400541,...,32,982.0,0.0,982,982,83500000.0,0.0,83500000,83500000,DoS Hulk
468552,53,47600,1,1,61,156,61,61,61.00,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


**Reasoning**:
Identify and handle missing values in the sampled DataFrame.



In [12]:
nan_counts = df_sampled.isnull().sum()
nan_counts = nan_counts[nan_counts > 0]
display(nan_counts)

if not nan_counts.empty:
    for col in nan_counts.index:
        if df_sampled[col].dtype in ['int64', 'float64']:
            mean_val = df_sampled[col].mean()
            df_sampled[col].fillna(mean_val, inplace=True)
            print(f"Filled missing values in column '{col}' with mean: {mean_val}")
        else:
            # For categorical columns, fill with mode or a placeholder
            mode_val = df_sampled[col].mode()[0]
            df_sampled[col].fillna(mode_val, inplace=True)
            print(f"Filled missing values in column '{col}' with mode: {mode_val}")

nan_counts_after = df_sampled.isnull().sum()
nan_counts_after = nan_counts_after[nan_counts_after > 0]
display(nan_counts_after)

,0


,0


**Reasoning**:
Identify and one-hot encode the categorical columns in the sampled DataFrame, then scale the numerical features, and store the processed data in a new DataFrame.



In [17]:
import numpy as np
from sklearn.preprocessing import StandardScaler

categorical_cols = df_sampled.select_dtypes(include=['object']).columns
df_processed = pd.get_dummies(df_sampled, columns=categorical_cols, drop_first=True)

numerical_cols = df_processed.select_dtypes(include=['int64', 'float64']).columns

# Handle infinite or too large values
for col in numerical_cols:
    if np.isinf(df_processed[col]).any() or (df_processed[col] > np.finfo(np.float64).max).any():
        print(f"Column '{col}' contains infinity or values too large for float64.")
        df_processed[col] = df_processed[col].replace([np.inf, -np.inf], np.nan)
        mean_val = df_processed[col].mean()
        df_processed[col].fillna(mean_val, inplace=True)
        print(f"Replaced infinity with NaN and filled NaN with mean in column '{col}'.")


scaler = StandardScaler()
df_processed[numerical_cols] = scaler.fit_transform(df_processed[numerical_cols])

display(df_processed.head())

Column 'Flow Bytes/s' contains infinity or values too large for float64.
Replaced infinity with NaN and filled NaN with mean in column 'Flow Bytes/s'.
Column ' Flow Packets/s' contains infinity or values too large for float64.
Replaced infinity with NaN and filled NaN with mean in column ' Flow Packets/s'.


/tmp/ipython-input-2216581541.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_processed[col].fillna(mean_val, inplace=True)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label_DoS GoldenEye,Label_DoS Hulk,Label_DoS Slowhttptest,Label_DoS slowloris,Label_Heartbleed
15364,-0.357853,-0.652606,-0.011370,-0.010111,-0.068978,-0.008492,-0.311170,0.473831,-0.125422,-0.359254,...,-0.108095,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False
18606,-0.333049,-0.624223,-0.003220,-0.003550,0.031294,-0.007542,0.047435,-0.282394,0.148193,0.069534,...,-0.108095,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False
454868,-0.357853,-0.654524,-0.011370,-0.010111,-0.066131,-0.008496,-0.274675,0.898054,0.017279,-0.359254,...,-0.108095,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False
123291,-0.356135,1.300240,-0.005031,-0.006466,-0.028745,-0.005253,0.204524,-0.282394,-0.095951,0.196319,...,-0.106481,1.613253,-0.105996,1.587243,1.625364,False,True,False,False,False
468552,-0.357853,-0.653970,-0.011370,-0.010111,-0.066502,-0.008485,-0.279435,0.842721,-0.001334,-0.359254,...,-0.108095,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False


In [25]:
display(df_processed.head())

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Idle Mean,Idle Std,Idle Max,Idle Min,Label_DoS GoldenEye,Label_DoS Hulk,Label_DoS Slowhttptest,Label_DoS slowloris,Label_Heartbleed,anomaly_score
15364,-0.357853,-0.652606,-0.011370,-0.010111,-0.068978,-0.008492,-0.311170,0.473831,-0.125422,-0.359254,...,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False,1
18606,-0.333049,-0.624223,-0.003220,-0.003550,0.031294,-0.007542,0.047435,-0.282394,0.148193,0.069534,...,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False,1
454868,-0.357853,-0.654524,-0.011370,-0.010111,-0.066131,-0.008496,-0.274675,0.898054,0.017279,-0.359254,...,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False,1
123291,-0.356135,1.300240,-0.005031,-0.006466,-0.028745,-0.005253,0.204524,-0.282394,-0.095951,0.196319,...,1.613253,-0.105996,1.587243,1.625364,False,True,False,False,False,1
468552,-0.357853,-0.653970,-0.011370,-0.010111,-0.066502,-0.008485,-0.279435,0.842721,-0.001334,-0.359254,...,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False,1


**Reasoning**:
The scaling failed because the data contains infinity or values too large for float64. I need to inspect the numerical columns for such values and handle them before scaling.



In [14]:
import numpy as np
from sklearn.preprocessing import StandardScaler

numerical_cols = df_processed.select_dtypes(include=['int64', 'float64']).columns
for col in numerical_cols:
    if np.isinf(df_processed[col]).any() or (df_processed[col] > np.finfo(np.float64).max).any():
        print(f"Column '{col}' contains infinity or values too large for float64.")
        # Replace infinite values with a large but finite number or NaN, then handle NaN
        df_processed[col] = df_processed[col].replace([np.inf, -np.inf], np.nan)
        mean_val = df_processed[col].mean()
        df_processed[col].fillna(mean_val, inplace=True)
        print(f"Replaced infinity with NaN and filled NaN with mean in column '{col}'.")

scaler = StandardScaler()
df_processed[numerical_cols] = scaler.fit_transform(df_processed[numerical_cols])

display(df_processed.head())

Column 'Flow Bytes/s' contains infinity or values too large for float64.
Replaced infinity with NaN and filled NaN with mean in column 'Flow Bytes/s'.
Column ' Flow Packets/s' contains infinity or values too large for float64.
Replaced infinity with NaN and filled NaN with mean in column ' Flow Packets/s'.


/tmp/ipython-input-726201900.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_processed[col].fillna(mean_val, inplace=True)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label_DoS GoldenEye,Label_DoS Hulk,Label_DoS Slowhttptest,Label_DoS slowloris,Label_Heartbleed
15364,-0.357853,-0.652606,-0.011370,-0.010111,-0.068978,-0.008492,-0.311170,0.473831,-0.125422,-0.359254,...,-0.108095,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False
18606,-0.333049,-0.624223,-0.003220,-0.003550,0.031294,-0.007542,0.047435,-0.282394,0.148193,0.069534,...,-0.108095,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False
454868,-0.357853,-0.654524,-0.011370,-0.010111,-0.066131,-0.008496,-0.274675,0.898054,0.017279,-0.359254,...,-0.108095,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False
123291,-0.356135,1.300240,-0.005031,-0.006466,-0.028745,-0.005253,0.204524,-0.282394,-0.095951,0.196319,...,-0.106481,1.613253,-0.105996,1.587243,1.625364,False,True,False,False,False
468552,-0.357853,-0.653970,-0.011370,-0.010111,-0.066502,-0.008485,-0.279435,0.842721,-0.001334,-0.359254,...,-0.108095,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False


K MEANS CLUSTERING

In [15]:
from sklearn.cluster import KMeans

n_clusters = 6
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
kmeans.fit(df_processed)

df_processed['cluster_label'] = kmeans.labels_

display(df_processed.head())

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Idle Mean,Idle Std,Idle Max,Idle Min,Label_DoS GoldenEye,Label_DoS Hulk,Label_DoS Slowhttptest,Label_DoS slowloris,Label_Heartbleed,cluster_label
15364,-0.357853,-0.652606,-0.011370,-0.010111,-0.068978,-0.008492,-0.311170,0.473831,-0.125422,-0.359254,...,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False,0
18606,-0.333049,-0.624223,-0.003220,-0.003550,0.031294,-0.007542,0.047435,-0.282394,0.148193,0.069534,...,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False,5
454868,-0.357853,-0.654524,-0.011370,-0.010111,-0.066131,-0.008496,-0.274675,0.898054,0.017279,-0.359254,...,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False,0
123291,-0.356135,1.300240,-0.005031,-0.006466,-0.028745,-0.005253,0.204524,-0.282394,-0.095951,0.196319,...,1.613253,-0.105996,1.587243,1.625364,False,True,False,False,False,1
468552,-0.357853,-0.653970,-0.011370,-0.010111,-0.066502,-0.008485,-0.279435,0.842721,-0.001334,-0.359254,...,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False,0


**Reasoning**:
Extract the true labels from the original sampled dataframe, calculate the Adjusted Rand Index (ARI) between the true labels and the cluster labels assigned by k-means, and display the ARI score.



In [16]:
from sklearn.metrics import adjusted_rand_score

true_labels = df_sampled[' Label']
ari_score = adjusted_rand_score(true_labels, df_processed['cluster_label'])

print(f"Adjusted Rand Index (ARI): {ari_score}")

Adjusted Rand Index (ARI): 0.3299139054010561


ISOLATION FOREST

**Reasoning**:
Check if the sampled DataFrame `df_sampled` exists. If not, sample 10% of the original DataFrame and display its head.



In [18]:
if 'df_sampled' not in locals():
    df_sampled = df.sample(frac=0.1, random_state=42)

display(df_sampled.head())

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
15364,53,105939,1,1,41,131,41,41,41.00,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
18606,443,1319967,10,10,851,3492,267,0,85.10,99.870416,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
454868,53,23894,1,1,64,117,64,64,64.00,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
123291,80,83635124,8,6,366,11595,366,0,45.75,129.400541,...,32,982.0,0.0,982,982,83500000.0,0.0,83500000,83500000,DoS Hulk
468552,53,47600,1,1,61,156,61,61,61.00,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


**Reasoning**:
Handle missing values by filling numerical columns with the mean and categorical columns with the mode.



In [19]:
nan_counts = df_sampled.isnull().sum()
nan_counts = nan_counts[nan_counts > 0]
display(nan_counts)

if not nan_counts.empty:
    for col in nan_counts.index:
        if df_sampled[col].dtype in ['int64', 'float64']:
            mean_val = df_sampled[col].mean()
            df_sampled[col].fillna(mean_val, inplace=True)
            print(f"Filled missing values in column '{col}' with mean: {mean_val}")
        else:
            mode_val = df_sampled[col].mode()[0]
            df_sampled[col].fillna(mode_val, inplace=True)
            print(f"Filled missing values in column '{col}' with mode: {mode_val}")

nan_counts_after = df_sampled.isnull().sum()
nan_counts_after = nan_counts_after[nan_counts_after > 0]
display(nan_counts_after)

,0


,0


**Reasoning**:
One-hot encode the categorical features, handle infinite values in numerical columns, and scale the numerical features for Isolation Forest.



In [20]:
import numpy as np
from sklearn.preprocessing import StandardScaler

categorical_cols = df_sampled.select_dtypes(include=['object']).columns
df_processed = pd.get_dummies(df_sampled, columns=categorical_cols, drop_first=True)

numerical_cols = df_processed.select_dtypes(include=['int64', 'float64']).columns

for col in numerical_cols:
    if np.isinf(df_processed[col]).any() or (df_processed[col] > np.finfo(np.float64).max).any():
        print(f"Column '{col}' contains infinity or values too large for float64.")
        df_processed[col] = df_processed[col].replace([np.inf, -np.inf], np.nan)
        mean_val = df_processed[col].mean()
        df_processed[col].fillna(mean_val, inplace=True)
        print(f"Replaced infinity with NaN and filled NaN with mean in column '{col}'.")


scaler = StandardScaler()
df_processed[numerical_cols] = scaler.fit_transform(df_processed[numerical_cols])

display(df_processed.head())

Column 'Flow Bytes/s' contains infinity or values too large for float64.
Replaced infinity with NaN and filled NaN with mean in column 'Flow Bytes/s'.
Column ' Flow Packets/s' contains infinity or values too large for float64.
Replaced infinity with NaN and filled NaN with mean in column ' Flow Packets/s'.


/tmp/ipython-input-4167125401.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_processed[col].fillna(mean_val, inplace=True)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label_DoS GoldenEye,Label_DoS Hulk,Label_DoS Slowhttptest,Label_DoS slowloris,Label_Heartbleed
15364,-0.357853,-0.652606,-0.011370,-0.010111,-0.068978,-0.008492,-0.311170,0.473831,-0.125422,-0.359254,...,-0.108095,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False
18606,-0.333049,-0.624223,-0.003220,-0.003550,0.031294,-0.007542,0.047435,-0.282394,0.148193,0.069534,...,-0.108095,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False
454868,-0.357853,-0.654524,-0.011370,-0.010111,-0.066131,-0.008496,-0.274675,0.898054,0.017279,-0.359254,...,-0.108095,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False
123291,-0.356135,1.300240,-0.005031,-0.006466,-0.028745,-0.005253,0.204524,-0.282394,-0.095951,0.196319,...,-0.106481,1.613253,-0.105996,1.587243,1.625364,False,True,False,False,False
468552,-0.357853,-0.653970,-0.011370,-0.010111,-0.066502,-0.008485,-0.279435,0.842721,-0.001334,-0.359254,...,-0.108095,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False


**Reasoning**:
Train an Isolation Forest model on the processed data.



In [21]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(n_estimators=100, random_state=42, contamination='auto')
model.fit(df_processed)

IsolationForest(random_state=42)

**Reasoning**:
Predict anomalies using the trained Isolation Forest model and store the results in a new column.



In [22]:
df_processed['anomaly_score'] = model.predict(df_processed)

display(df_processed.head())

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Idle Mean,Idle Std,Idle Max,Idle Min,Label_DoS GoldenEye,Label_DoS Hulk,Label_DoS Slowhttptest,Label_DoS slowloris,Label_Heartbleed,anomaly_score
15364,-0.357853,-0.652606,-0.011370,-0.010111,-0.068978,-0.008492,-0.311170,0.473831,-0.125422,-0.359254,...,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False,1
18606,-0.333049,-0.624223,-0.003220,-0.003550,0.031294,-0.007542,0.047435,-0.282394,0.148193,0.069534,...,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False,1
454868,-0.357853,-0.654524,-0.011370,-0.010111,-0.066131,-0.008496,-0.274675,0.898054,0.017279,-0.359254,...,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False,1
123291,-0.356135,1.300240,-0.005031,-0.006466,-0.028745,-0.005253,0.204524,-0.282394,-0.095951,0.196319,...,1.613253,-0.105996,1.587243,1.625364,False,True,False,False,False,1
468552,-0.357853,-0.653970,-0.011370,-0.010111,-0.066502,-0.008485,-0.279435,0.842721,-0.001334,-0.359254,...,-0.579740,-0.105996,-0.584989,-0.570416,False,False,False,False,False,1


**Reasoning**:
Extract the true labels, create binary true labels, calculate and display classification metrics for the anomaly detection.



In [23]:
from sklearn.metrics import classification_report, confusion_matrix

true_labels = df_sampled[' Label']

# Map true labels to binary: 1 for BENIGN (normal), -1 for others (anomaly)
binary_true_labels = true_labels.apply(lambda x: 1 if x == 'BENIGN' else -1)

# Get the predicted anomaly labels from the 'anomaly_score' column
predicted_labels = df_processed['anomaly_score']

# Calculate and print classification report
print("Classification Report:")
print(classification_report(binary_true_labels, predicted_labels))

# Calculate and print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(binary_true_labels, predicted_labels))

Classification Report:
              precision    recall  f1-score   support

          -1       0.36      0.06      0.10     25037
           1       0.64      0.94      0.76     44133

    accuracy                           0.62     69170
   macro avg       0.50      0.50      0.43     69170
weighted avg       0.54      0.62      0.52     69170

Confusion Matrix:
[[ 1441 23596]
 [ 2526 41607]]


COMPARE AND FIND BEST ONE

**Reasoning**:
Recall and display the evaluation metrics for K-Means (ARI) and Isolation Forest (Classification Report and Confusion Matrix) from previous steps to summarize their performance for anomaly detection.



In [26]:
print("K-Means Clustering Evaluation:")
print(f"Adjusted Rand Index (ARI): {ari_score}")
print("\nIsolation Forest Evaluation:")
print("Classification Report:")
print(classification_report(binary_true_labels, predicted_labels))
print("Confusion Matrix:")
print(confusion_matrix(binary_true_labels, predicted_labels))

K-Means Clustering Evaluation:
Adjusted Rand Index (ARI): 0.3299139054010561

Isolation Forest Evaluation:
Classification Report:
              precision    recall  f1-score   support

          -1       0.36      0.06      0.10     25037
           1       0.64      0.94      0.76     44133

    accuracy                           0.62     69170
   macro avg       0.50      0.50      0.43     69170
weighted avg       0.54      0.62      0.52     69170

Confusion Matrix:
[[ 1441 23596]
 [ 2526 41607]]


**Reasoning**:
Identify other relevant models for anomaly detection or clustering based on the nature of the dataset and the previous models used.



In [27]:
print("Other relevant models for anomaly detection or clustering could include:")
print("- DBSCAN: A density-based clustering algorithm that can find arbitrarily shaped clusters and identify noise (anomalies). It's suitable for data that doesn't have well-defined spherical clusters, which might be the case for network traffic data.")
print("- Local Outlier Factor (LOF): An unsupervised anomaly detection method that computes the local deviation of density of a given data point with respect to its neighbors. It can be effective in identifying outliers in datasets where the anomaly is relative to the local density rather than the global data distribution.")

Other relevant models for anomaly detection or clustering could include:
- DBSCAN: A density-based clustering algorithm that can find arbitrarily shaped clusters and identify noise (anomalies). It's suitable for data that doesn't have well-defined spherical clusters, which might be the case for network traffic data.
- Local Outlier Factor (LOF): An unsupervised anomaly detection method that computes the local deviation of density of a given data point with respect to its neighbors. It can be effective in identifying outliers in datasets where the anomaly is relative to the local density rather than the global data distribution.


**Reasoning**:
Apply DBSCAN and LOF models, evaluate their performance, and display the results.



In [28]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import adjusted_rand_score, classification_report, confusion_matrix

# Apply DBSCAN
# Choose eps and min_samples - these often require tuning.
# Starting with values that might be too broad but provide initial clusters.
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(df_processed)

# Evaluate DBSCAN
# DBSCAN labels -1 indicate noise (anomalies)
ari_dbscan = adjusted_rand_score(binary_true_labels, dbscan_labels)
print(f"DBSCAN Adjusted Rand Index (ARI): {ari_dbscan}")

# Apply Local Outlier Factor (LOF)
# Choose n_neighbors - usually between 20 and 100
lof = LocalOutlierFactor(n_neighbors=20, contamination='auto')
# Fit and predict on the data
lof_predictions = lof.fit_predict(df_processed)

# Evaluate LOF
print("\nLocal Outlier Factor (LOF) Evaluation:")
print("Classification Report:")
print(classification_report(binary_true_labels, lof_predictions))

print("Confusion Matrix:")
print(confusion_matrix(binary_true_labels, lof_predictions))

DBSCAN Adjusted Rand Index (ARI): 0.07720080696811063

Local Outlier Factor (LOF) Evaluation:
Classification Report:
              precision    recall  f1-score   support

          -1       0.33      0.14      0.19     25037
           1       0.63      0.84      0.72     44133

    accuracy                           0.59     69170
   macro avg       0.48      0.49      0.46     69170
weighted avg       0.52      0.59      0.53     69170

Confusion Matrix:
[[ 3422 21615]
 [ 6856 37277]]


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


**Reasoning**:
Summarize the performance metrics of all applied models (K-Means, Isolation Forest, DBSCAN, and LOF) in a markdown cell for comparison.



In [29]:
# Display the summary of evaluation metrics for all models
print("--- Model Performance Comparison ---")

print("\nK-Means Clustering Evaluation:")
print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")

print("\nIsolation Forest Evaluation:")
print("Classification Report:")
print(classification_report(binary_true_labels, predicted_labels))
print("Confusion Matrix:")
print(confusion_matrix(binary_true_labels, predicted_labels))

print("\nDBSCAN Clustering Evaluation:")
print(f"Adjusted Rand Index (ARI): {ari_dbscan:.4f}")

print("\nLocal Outlier Factor (LOF) Evaluation:")
print("Classification Report:")
print(classification_report(binary_true_labels, lof_predictions))
print("Confusion Matrix:")
print(confusion_matrix(binary_true_labels, lof_predictions))

--- Model Performance Comparison ---

K-Means Clustering Evaluation:
Adjusted Rand Index (ARI): 0.3299

Isolation Forest Evaluation:
Classification Report:
              precision    recall  f1-score   support

          -1       0.36      0.06      0.10     25037
           1       0.64      0.94      0.76     44133

    accuracy                           0.62     69170
   macro avg       0.50      0.50      0.43     69170
weighted avg       0.54      0.62      0.52     69170

Confusion Matrix:
[[ 1441 23596]
 [ 2526 41607]]

DBSCAN Clustering Evaluation:
Adjusted Rand Index (ARI): 0.0772

Local Outlier Factor (LOF) Evaluation:
Classification Report:
              precision    recall  f1-score   support

          -1       0.33      0.14      0.19     25037
           1       0.63      0.84      0.72     44133

    accuracy                           0.59     69170
   macro avg       0.48      0.49      0.46     69170
weighted avg       0.52      0.59      0.53     69170

Confusion Mat

**Reasoning**:
The evaluation metrics for all models have been presented. Now, create a markdown summary to compare the performance of each model based on these metrics.



In [30]:
# Create a markdown cell with the comparison summary
from IPython.display import display, Markdown

summary_markdown = """
## Model Performance Comparison Summary

Here's a comparison of the performance metrics for the applied anomaly detection and clustering models:

| Model            | Metric                                  | Score/Value                                  | Notes                                                                                                |
|------------------|-----------------------------------------|----------------------------------------------|------------------------------------------------------------------------------------------------------|
| **K-Means**      | Adjusted Rand Index (ARI)               | {ari_score:.4f}                              | Measures similarity between clusters and true labels. Moderate agreement.                             |
| **Isolation Forest**| Precision (Anomaly -1)                  | 0.36                                         | Low precision in identifying anomalies.                                                              |
|                  | Recall (Anomaly -1)                     | 0.06                                         | Very low recall, missed a large number of actual anomalies.                                          |
|                  | F1-score (Anomaly -1)                   | 0.10                                         | Poor overall performance for the anomaly class.                                                      |
|                  | Confusion Matrix                        | [[1441, 23596], [2526, 41607]]               | High number of False Positives (23596) and False Negatives (2526) for anomalies.                     |
| **DBSCAN**       | Adjusted Rand Index (ARI)               | {ari_dbscan:.4f}                             | Very low ARI, poor agreement with true labels.                                                       |
| **Local Outlier Factor (LOF)** | Precision (Anomaly -1)        | 0.33                                         | Low precision in identifying anomalies.                                                              |
|                  | Recall (Anomaly -1)                     | 0.14                                         | Low recall, missed many actual anomalies.                                                            |
|                  | F1-score (Anomaly -1)                   | 0.19                                         | Poor overall performance for the anomaly class.                                                      |
|                  | Confusion Matrix                        | [[3422, 21615], [6856, 37277]]               | High number of False Positives (21615) and False Negatives (6856) for anomalies.                     |

### Analysis:

*   **K-Means** showed a moderate ARI score, suggesting some level of grouping that aligns with the true labels, although it is not a direct anomaly detection method in this application.
*   Both **Isolation Forest** and **Local Outlier Factor (LOF)**, which are designed for anomaly detection, performed poorly in identifying the anomaly class (-1) based on the standard classification metrics (precision, recall, F1-score). They had low recall, indicating they missed a significant portion of the actual anomalies.
*   **DBSCAN** also showed very poor performance in terms of ARI, suggesting that the chosen parameters did not result in clusters or outliers that align well with the true labels.

### Conclusion:

Based on the evaluated metrics, none of the applied models with their current configurations performed particularly well in effectively identifying anomalies in this dataset sample when compared to the provided true labels. The Isolation Forest and LOF models, despite being anomaly detection techniques, struggled significantly with recall for the anomaly class. Further hyperparameter tuning, exploring alternative anomaly detection algorithms, or re-evaluating the definition of "anomaly" in the context of this dataset might be necessary to achieve better results.
""".format(ari_score=ari_score, ari_dbscan=ari_dbscan)

display(Markdown(summary_markdown))


## Model Performance Comparison Summary

Here's a comparison of the performance metrics for the applied anomaly detection and clustering models:

| Model            | Metric                                  | Score/Value                                  | Notes                                                                                                |
|------------------|-----------------------------------------|----------------------------------------------|------------------------------------------------------------------------------------------------------|
| **K-Means**      | Adjusted Rand Index (ARI)               | 0.3299                              | Measures similarity between clusters and true labels. Moderate agreement.                             |
| **Isolation Forest**| Precision (Anomaly -1)                  | 0.36                                         | Low precision in identifying anomalies.                                                              |
|                  | Recall (Anomaly -1)                     | 0.06                                         | Very low recall, missed a large number of actual anomalies.                                          |
|                  | F1-score (Anomaly -1)                   | 0.10                                         | Poor overall performance for the anomaly class.                                                      |
|                  | Confusion Matrix                        | [[1441, 23596], [2526, 41607]]               | High number of False Positives (23596) and False Negatives (2526) for anomalies.                     |
| **DBSCAN**       | Adjusted Rand Index (ARI)               | 0.0772                             | Very low ARI, poor agreement with true labels.                                                       |
| **Local Outlier Factor (LOF)** | Precision (Anomaly -1)        | 0.33                                         | Low precision in identifying anomalies.                                                              |
|                  | Recall (Anomaly -1)                     | 0.14                                         | Low recall, missed many actual anomalies.                                                            |
|                  | F1-score (Anomaly -1)                   | 0.19                                         | Poor overall performance for the anomaly class.                                                      |
|                  | Confusion Matrix                        | [[3422, 21615], [6856, 37277]]               | High number of False Positives (21615) and False Negatives (6856) for anomalies.                     |

### Analysis:

*   **K-Means** showed a moderate ARI score, suggesting some level of grouping that aligns with the true labels, although it is not a direct anomaly detection method in this application.
*   Both **Isolation Forest** and **Local Outlier Factor (LOF)**, which are designed for anomaly detection, performed poorly in identifying the anomaly class (-1) based on the standard classification metrics (precision, recall, F1-score). They had low recall, indicating they missed a significant portion of the actual anomalies.
*   **DBSCAN** also showed very poor performance in terms of ARI, suggesting that the chosen parameters did not result in clusters or outliers that align well with the true labels.

### Conclusion:

Based on the evaluated metrics, none of the applied models with their current configurations performed particularly well in effectively identifying anomalies in this dataset sample when compared to the provided true labels. The Isolation Forest and LOF models, despite being anomaly detection techniques, struggled significantly with recall for the anomaly class. Further hyperparameter tuning, exploring alternative anomaly detection algorithms, or re-evaluating the definition of "anomaly" in the context of this dataset might be necessary to achieve better results.


**Reasoning**:
Review the model performance comparison summary and summarize the comparison process and the rationale for selecting the best model in a markdown cell.



In [31]:
from IPython.display import display, Markdown

summary_text = """
## Model Comparison Summary and Selection Rationale

This analysis compared the performance of four models—K-Means, Isolation Forest, DBSCAN, and Local Outlier Factor (LOF)—on a 10% sample of the dataset for the purpose of anomaly detection.

The key metrics used for comparison were:

*   **Adjusted Rand Index (ARI):** Used for clustering methods (K-Means, DBSCAN) to measure the similarity between the model's clusters and the true labels. A higher ARI indicates better agreement.
*   **Precision, Recall, and F1-score:** Used for anomaly detection methods (Isolation Forest, LOF) to evaluate their effectiveness in classifying instances as normal or anomalous. These metrics were specifically examined for the anomaly class (-1).
*   **Confusion Matrix:** Provided a detailed breakdown of true positives, true negatives, false positives, and false negatives, offering deeper insight into the models' classification performance.

**Performance of Each Model:**

*   **K-Means:** Achieved an ARI of approximately 0.33. This suggests a moderate level of agreement between the clusters formed by K-Means and the true anomaly labels. However, K-Means is primarily a clustering algorithm and its effectiveness for direct anomaly detection depends on how well anomalies form distinct clusters.
*   **Isolation Forest:** Showed poor performance in identifying anomalies, with a precision of 0.36, recall of 0.06, and an F1-score of 0.10 for the anomaly class. The confusion matrix indicated a high number of false positives and false negatives for anomalies.
*   **DBSCAN:** Had a very low ARI of approximately 0.08. This suggests that with the chosen parameters, DBSCAN's density-based clustering did not align well with the true anomaly distribution.
*   **Local Outlier Factor (LOF):** Also demonstrated poor performance in anomaly detection, with a precision of 0.33, recall of 0.14, and an F1-score of 0.19 for the anomaly class. Similar to Isolation Forest, the confusion matrix revealed a significant number of misclassifications for anomalies.

**Rationale for Model Selection:**

Based on the evaluation metrics, **none of the applied models with their current configurations performed well in effectively identifying anomalies** in this dataset sample when compared to the provided true labels. The anomaly detection methods (Isolation Forest and LOF) struggled significantly with recall, meaning they failed to identify a large proportion of the actual anomalies. The clustering methods (K-Means and DBSCAN), while providing some structural insights (moderate ARI for K-Means), did not translate into effective anomaly detection based on the evaluation against the binary true labels.

Therefore, **no single model can be confidently selected as the "best"** for this task with the current results. Further hyperparameter tuning for all models, exploring alternative anomaly detection algorithms better suited for the characteristics of this dataset, or refining the preprocessing steps are necessary next steps to improve performance. The current results indicate that the anomaly detection problem on this dataset is challenging for these standard methods without further optimization.
"""

display(Markdown(summary_text))


## Model Comparison Summary and Selection Rationale

This analysis compared the performance of four models—K-Means, Isolation Forest, DBSCAN, and Local Outlier Factor (LOF)—on a 10% sample of the dataset for the purpose of anomaly detection.

The key metrics used for comparison were:

*   **Adjusted Rand Index (ARI):** Used for clustering methods (K-Means, DBSCAN) to measure the similarity between the model's clusters and the true labels. A higher ARI indicates better agreement.
*   **Precision, Recall, and F1-score:** Used for anomaly detection methods (Isolation Forest, LOF) to evaluate their effectiveness in classifying instances as normal or anomalous. These metrics were specifically examined for the anomaly class (-1).
*   **Confusion Matrix:** Provided a detailed breakdown of true positives, true negatives, false positives, and false negatives, offering deeper insight into the models' classification performance.

**Performance of Each Model:**

*   **K-Means:** Achieved an ARI of approximately 0.33. This suggests a moderate level of agreement between the clusters formed by K-Means and the true anomaly labels. However, K-Means is primarily a clustering algorithm and its effectiveness for direct anomaly detection depends on how well anomalies form distinct clusters.
*   **Isolation Forest:** Showed poor performance in identifying anomalies, with a precision of 0.36, recall of 0.06, and an F1-score of 0.10 for the anomaly class. The confusion matrix indicated a high number of false positives and false negatives for anomalies.
*   **DBSCAN:** Had a very low ARI of approximately 0.08. This suggests that with the chosen parameters, DBSCAN's density-based clustering did not align well with the true anomaly distribution.
*   **Local Outlier Factor (LOF):** Also demonstrated poor performance in anomaly detection, with a precision of 0.33, recall of 0.14, and an F1-score of 0.19 for the anomaly class. Similar to Isolation Forest, the confusion matrix revealed a significant number of misclassifications for anomalies.

**Rationale for Model Selection:**

Based on the evaluation metrics, **none of the applied models with their current configurations performed well in effectively identifying anomalies** in this dataset sample when compared to the provided true labels. The anomaly detection methods (Isolation Forest and LOF) struggled significantly with recall, meaning they failed to identify a large proportion of the actual anomalies. The clustering methods (K-Means and DBSCAN), while providing some structural insights (moderate ARI for K-Means), did not translate into effective anomaly detection based on the evaluation against the binary true labels.

Therefore, **no single model can be confidently selected as the "best"** for this task with the current results. Further hyperparameter tuning for all models, exploring alternative anomaly detection algorithms better suited for the characteristics of this dataset, or refining the preprocessing steps are necessary next steps to improve performance. The current results indicate that the anomaly detection problem on this dataset is challenging for these standard methods without further optimization.


## Tune hyperparameters using cross validation

Tune the hyperparameters of the promising models using cross-validation.


**Reasoning**:
Identify promising models, define hyperparameter grids, and set up GridSearchCV for Isolation Forest and LOF. Evaluate the models with the tuned parameters.



In [37]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import make_scorer, f1_score, recall_score, precision_score, classification_report, confusion_matrix
import numpy as np

# Define scoring metrics for anomaly detection, focusing on the anomaly class (-1)
# Using make_scorer to create scorers that specifically evaluate the -1 class
f1_scorer = make_scorer(f1_score, pos_label=-1)
recall_scorer = make_scorer(recall_score, pos_label=-1)
precision_scorer = make_scorer(precision_score, pos_label=-1)

scoring = {'f1': f1_scorer, 'recall': recall_scorer, 'precision': precision_scorer}


# Define hyperparameter grid for Isolation Forest
isolation_forest_param_grid = {
    'n_estimators': [50, 100, 200],
    'contamination': ['auto', 0.01, 0.05, 0.1]
}

# Define hyperparameter grid for Local Outlier Factor
lof_param_grid = {
    'n_neighbors': [20, 50, 100],
    'contamination': ['auto', 0.01, 0.05, 0.1]
}

# Initialize GridSearchCV for Isolation Forest
isolation_forest_grid_search = GridSearchCV(
    IsolationForest(random_state=42),
    isolation_forest_param_grid,
    scoring=scoring,
    refit='f1', # Refit the model using the best f1 score
    cv=3, # Using 3-fold cross-validation
    n_jobs=-1 # Use all available cores
)

# Fit GridSearchCV for Isolation Forest
print("Tuning Isolation Forest...")
isolation_forest_grid_search.fit(df_processed, binary_true_labels)

# Get the best Isolation Forest model and hyperparameters
best_isolation_forest = isolation_forest_grid_search.best_estimator_
best_isolation_forest_params = isolation_forest_grid_search.best_params_

print("\nBest Isolation Forest Hyperparameters:", best_isolation_forest_params)

# Evaluate the best Isolation Forest model on the entire processed data
isolation_forest_predictions = best_isolation_forest.predict(df_processed)

print("\nFine-tuned Isolation Forest Evaluation:")
print("Classification Report:")
print(classification_report(binary_true_labels, isolation_forest_predictions))
print("Confusion Matrix:")
print(confusion_matrix(binary_true_labels, isolation_forest_predictions))


# Define a custom scorer for LOF using fit_predict
def lof_scorer(estimator, X, y_true):
    y_pred = estimator.fit_predict(X)
    return f1_score(y_true, y_pred, pos_label=-1)

# Initialize GridSearchCV for Local Outlier Factor with the custom scorer
lof_grid_search = GridSearchCV(
    LocalOutlierFactor(novelty=False), # novelty=False for unsupervised outlier detection
    lof_param_grid,
    scoring={'f1_lof': lof_scorer}, # Use the custom scorer
    refit='f1_lof', # Refit the model using the best custom f1 score
    cv=3, # Using 3-fold cross-validation
    n_jobs=-1 # Use all available cores
)

# Fit GridSearchCV for Local Outlier Factor
print("\nTuning Local Outlier Factor (LOF)...")
lof_grid_search.fit(df_processed, binary_true_labels)

# Get the best LOF model and hyperparameters
best_lof = lof_grid_search.best_estimator_
best_lof_params = lof_grid_search.best_params_

print("\nBest Local Outlier Factor (LOF) Hyperparameters:", best_lof_params)

# Evaluate the best LOF model on the entire processed data
lof_predictions_tuned = best_lof.fit_predict(df_processed) # fit_predict for LOF unsupervised

print("\nFine-tuned Local Outlier Factor (LOF) Evaluation:")
print("Classification Report:")
print(classification_report(binary_true_labels, lof_predictions_tuned))
print("Confusion Matrix:")
print(confusion_matrix(binary_true_labels, lof_predictions_tuned))

Tuning Isolation Forest...

Best Isolation Forest Hyperparameters: {'contamination': 0.1, 'n_estimators': 100}

Fine-tuned Isolation Forest Evaluation:
Classification Report:
              precision    recall  f1-score   support

          -1       0.47      0.13      0.20     25037
           1       0.65      0.92      0.76     44133

    accuracy                           0.63     69170
   macro avg       0.56      0.52      0.48     69170
weighted avg       0.59      0.63      0.56     69170

Confusion Matrix:
[[ 3264 21773]
 [ 3653 40480]]

Tuning Local Outlier Factor (LOF)...


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(



Best Local Outlier Factor (LOF) Hyperparameters: {'contamination': 'auto', 'n_neighbors': 50}

Fine-tuned Local Outlier Factor (LOF) Evaluation:
Classification Report:
              precision    recall  f1-score   support

          -1       0.34      0.14      0.20     25037
           1       0.63      0.84      0.72     44133

    accuracy                           0.59     69170
   macro avg       0.49      0.49      0.46     69170
weighted avg       0.53      0.59      0.53     69170

Confusion Matrix:
[[ 3608 21429]
 [ 7074 37059]]


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


**Reasoning**:
The LOF model in scikit-learn does not have a `predict` method when `novelty=False` (unsupervised mode), which is required by `GridSearchCV` for scoring. Instead, it has `fit_predict` and `decision_function`. I need to use a custom scorer or adapt the approach for LOF within `GridSearchCV`. Since LOF's primary output is the anomaly score (decision_function) and then a binary prediction (-1 or 1) via fit_predict, I will define a custom scorer that uses the `fit_predict` method and then calculates the desired metrics (F1, recall, precision) based on the true binary labels and the predicted labels.



In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import make_scorer, f1_score, recall_score, precision_score, classification_report, confusion_matrix
import numpy as np

# Define scoring metrics for anomaly detection, focusing on the anomaly class (-1)
# Using make_scorer to create scorers that specifically evaluate the -1 class
f1_scorer = make_scorer(f1_score, pos_label=-1)
recall_scorer = make_scorer(recall_score, pos_label=-1)
precision_scorer = make_scorer(precision_score, pos_label=-1)

scoring = {'f1': f1_scorer, 'recall': recall_scorer, 'precision': precision_scorer}


# Define hyperparameter grid for Isolation Forest
isolation_forest_param_grid = {
    'n_estimators': [50, 100, 200],
    'contamination': ['auto', 0.01, 0.05, 0.1]
}

# Define hyperparameter grid for Local Outlier Factor
lof_param_grid = {
    'n_neighbors': [20, 50, 100],
    'contamination': ['auto', 0.01, 0.05, 0.1]
}

# Initialize GridSearchCV for Isolation Forest
isolation_forest_grid_search = GridSearchCV(
    IsolationForest(random_state=42),
    isolation_forest_param_grid,
    scoring=scoring,
    refit='f1', # Refit the model using the best f1 score
    cv=3, # Using 3-fold cross-validation
    n_jobs=-1 # Use all available cores
)

# Fit GridSearchCV for Isolation Forest
print("Tuning Isolation Forest...")
isolation_forest_grid_search.fit(df_processed, binary_true_labels)

# Get the best Isolation Forest model and hyperparameters
best_isolation_forest = isolation_forest_grid_search.best_estimator_
best_isolation_forest_params = isolation_forest_grid_search.best_params_

print("\nBest Isolation Forest Hyperparameters:", best_isolation_forest_params)

# Evaluate the best Isolation Forest model on the entire processed data
isolation_forest_predictions = best_isolation_forest.predict(df_processed)

print("\nFine-tuned Isolation Forest Evaluation:")
print("Classification Report:")
print(classification_report(binary_true_labels, isolation_forest_predictions))
print("Confusion Matrix:")
print(confusion_matrix(binary_true_labels, isolation_forest_predictions))


# Define a custom scorer for LOF using fit_predict
def lof_scorer(estimator, X, y_true):
    y_pred = estimator.fit_predict(X)
    return f1_score(y_true, y_pred, pos_label=-1)

# Initialize GridSearchCV for Local Outlier Factor with the custom scorer
lof_grid_search = GridSearchCV(
    LocalOutlierFactor(novelty=False), # novelty=False for unsupervised outlier detection
    lof_param_grid,
    scoring={'f1_lof': lof_scorer}, # Use the custom scorer
    refit='f1_lof', # Refit the model using the best custom f1 score
    cv=3, # Using 3-fold cross-validation
    n_jobs=-1 # Use all available cores
)

# Fit GridSearchCV for Local Outlier Factor
print("\nTuning Local Outlier Factor (LOF)...")
lof_grid_search.fit(df_processed, binary_true_labels)

# Get the best LOF model and hyperparameters
best_lof = lof_grid_search.best_estimator_
best_lof_params = lof_grid_search.best_params_

print("\nBest Local Outlier Factor (LOF) Hyperparameters:", best_lof_params)

# Evaluate the best LOF model on the entire processed data
lof_predictions_tuned = best_lof.fit_predict(df_processed) # fit_predict for LOF unsupervised

print("\nFine-tuned Local Outlier Factor (LOF) Evaluation:")
print("Classification Report:")
print(classification_report(binary_true_labels, lof_predictions_tuned))
print("Confusion Matrix:")
print(confusion_matrix(binary_true_labels, lof_predictions_tuned))

Tuning Isolation Forest...

Best Isolation Forest Hyperparameters: {'contamination': 0.1, 'n_estimators': 100}

Fine-tuned Isolation Forest Evaluation:
Classification Report:
              precision    recall  f1-score   support

          -1       0.47      0.13      0.20     25037
           1       0.65      0.92      0.76     44133

    accuracy                           0.63     69170
   macro avg       0.56      0.52      0.48     69170
weighted avg       0.59      0.63      0.56     69170

Confusion Matrix:
[[ 3264 21773]
 [ 3653 40480]]

Tuning Local Outlier Factor (LOF)...


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(



Best Local Outlier Factor (LOF) Hyperparameters: {'contamination': 'auto', 'n_neighbors': 50}

Fine-tuned Local Outlier Factor (LOF) Evaluation:
Classification Report:
              precision    recall  f1-score   support

          -1       0.34      0.14      0.20     25037
           1       0.63      0.84      0.72     44133

    accuracy                           0.59     69170
   macro avg       0.49      0.49      0.46     69170
weighted avg       0.53      0.59      0.53     69170

Confusion Matrix:
[[ 3608 21429]
 [ 7074 37059]]


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


## Compare model performance

Compare the performance metrics of all applied models.


**Reasoning**:
Present a summary of the performance metrics for all models, including the tuned models and their best hyperparameters, in a clear format for comparison.



In [38]:
from IPython.display import display, Markdown
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

summary_markdown = """
## Model Performance Comparison Summary

Here's a comparison of the performance metrics for the applied anomaly detection and clustering models, including the results after hyperparameter tuning:

| Model                               | Metric                                  | Score/Value                                  | Notes                                                                                                |
|-------------------------------------|-----------------------------------------|----------------------------------------------|------------------------------------------------------------------------------------------------------|
| **K-Means (Untuned)**               | Adjusted Rand Index (ARI)               | {ari_score:.4f}                              | Measures similarity between clusters and true labels. Moderate agreement.                             |
| **Isolation Forest (Untuned)**      | Precision (Anomaly -1)                  | 0.36                                         | Low precision in identifying anomalies.                                                              |
|                                     | Recall (Anomaly -1)                     | 0.06                                         | Very low recall, missed a large number of actual anomalies.                                          |
|                                     | F1-score (Anomaly -1)                   | 0.10                                         | Poor overall performance for the anomaly class.                                                      |
|                                     | Confusion Matrix                        | [[1441, 23596], [2526, 41607]]               | High number of False Positives (23596) and False Negatives (2526) for anomalies.                     |
| **DBSCAN (Untuned)**                | Adjusted Rand Index (ARI)               | {ari_dbscan:.4f}                             | Very low ARI, poor agreement with true labels.                                                       |
| **Local Outlier Factor (LOF) (Untuned)** | Precision (Anomaly -1)        | 0.33                                         | Low precision in identifying anomalies.                                                              |
|                                     | Recall (Anomaly -1)                     | 0.14                                         | Low recall, missed many actual anomalies.                                                            |
|                                     | F1-score (Anomaly -1)                   | 0.19                                         | Poor overall performance for the anomaly class.                                                      |
|                                     | Confusion Matrix                        | [[3422, 21615], [6856, 37277]]               | High number of False Positives (21615) and False Negatives (6856) for anomalies.                     |
| **Isolation Forest (Tuned)**        | Best Hyperparameters                    | {best_isolation_forest_params}               | Hyperparameters found via GridSearchCV.                                                             |
|                                     | Precision (Anomaly -1)                  | {tuned_isolation_forest_precision:.4f}       | Precision for the anomaly class after tuning.                                                        |
|                                     | Recall (Anomaly -1)                     | {tuned_isolation_forest_recall:.4f}          | Recall for the anomaly class after tuning.                                                           |
|                                     | F1-score (Anomaly -1)                   | {tuned_isolation_forest_f1:.4f}              | F1-score for the anomaly class after tuning.                                                         |
|                                     | Confusion Matrix                        | {tuned_isolation_forest_confusion_matrix}    | Confusion matrix after tuning.                                                                       |
| **Local Outlier Factor (LOF) (Tuned)** | Best Hyperparameters          | {best_lof_params}                            | Hyperparameters found via GridSearchCV.                                                             |
|                                     | Precision (Anomaly -1)                  | {tuned_lof_precision:.4f}                    | Precision for the anomaly class after tuning.                                                        |
|                                     | Recall (Anomaly -1)                     | {tuned_lof_recall:.4f}                       | Recall for the anomaly class after tuning.                                                           |
|                                     | F1-score (Anomaly -1)                   | {tuned_lof_f1:.4f}                           | F1-score for the anomaly class after tuning.                                                         |
|                                     | Confusion Matrix                        | {tuned_lof_confusion_matrix}                 | Confusion matrix after tuning.                                                                       |

### Analysis:

*   The untuned Isolation Forest and LOF models showed poor performance in identifying anomalies, particularly in terms of recall.
*   Hyperparameter tuning improved the F1-scores for both Isolation Forest and LOF, but the overall performance in detecting anomalies remains relatively low, especially in terms of recall.
*   The ARI scores for K-Means and DBSCAN indicate limited agreement with the true anomaly labels, suggesting these clustering methods, with their current configurations, are not highly effective for direct anomaly detection in this dataset.
*   The tuned Isolation Forest and LOF models, while showing some improvement, still exhibit a significant number of false positives and false negatives for the anomaly class, highlighting the challenge of this anomaly detection task.

### Conclusion:

Even after hyperparameter tuning, the performance of Isolation Forest and Local Outlier Factor in identifying anomalies is not strong. The low recall scores for the anomaly class across these methods indicate that a large proportion of actual anomalies are being missed. The clustering methods (K-Means and DBSCAN) also did not demonstrate a strong alignment with the true anomaly labels based on ARI. Further efforts, potentially involving more extensive hyperparameter tuning, exploring different anomaly detection algorithms, or feature engineering, are likely required to achieve satisfactory anomaly detection performance on this dataset.
""".format(
    ari_score=ari_score,
    ari_dbscan=ari_dbscan,
    best_isolation_forest_params=best_isolation_forest_params,
    tuned_isolation_forest_precision=precision_score(binary_true_labels, isolation_forest_predictions, pos_label=-1),
    tuned_isolation_forest_recall=recall_score(binary_true_labels, isolation_forest_predictions, pos_label=-1),
    tuned_isolation_forest_f1=f1_score(binary_true_labels, isolation_forest_predictions, pos_label=-1),
    tuned_isolation_forest_confusion_matrix=confusion_matrix(binary_true_labels, isolation_forest_predictions).tolist(),
    best_lof_params=best_lof_params,
    tuned_lof_precision=precision_score(binary_true_labels, lof_predictions_tuned, pos_label=-1),
    tuned_lof_recall=recall_score(binary_true_labels, lof_predictions_tuned, pos_label=-1),
    tuned_lof_f1=f1_score(binary_true_labels, lof_predictions_tuned, pos_label=-1),
    tuned_lof_confusion_matrix=confusion_matrix(binary_true_labels, lof_predictions_tuned).tolist()
)

display(Markdown(summary_markdown))


## Model Performance Comparison Summary

Here's a comparison of the performance metrics for the applied anomaly detection and clustering models, including the results after hyperparameter tuning:

| Model                               | Metric                                  | Score/Value                                  | Notes                                                                                                |
|-------------------------------------|-----------------------------------------|----------------------------------------------|------------------------------------------------------------------------------------------------------|
| **K-Means (Untuned)**               | Adjusted Rand Index (ARI)               | 0.3299                              | Measures similarity between clusters and true labels. Moderate agreement.                             |
| **Isolation Forest (Untuned)**      | Precision (Anomaly -1)                  | 0.36                                         | Low precision in identifying anomalies.                                                              |
|                                     | Recall (Anomaly -1)                     | 0.06                                         | Very low recall, missed a large number of actual anomalies.                                          |
|                                     | F1-score (Anomaly -1)                   | 0.10                                         | Poor overall performance for the anomaly class.                                                      |
|                                     | Confusion Matrix                        | [[1441, 23596], [2526, 41607]]               | High number of False Positives (23596) and False Negatives (2526) for anomalies.                     |
| **DBSCAN (Untuned)**                | Adjusted Rand Index (ARI)               | 0.0772                             | Very low ARI, poor agreement with true labels.                                                       |
| **Local Outlier Factor (LOF) (Untuned)** | Precision (Anomaly -1)        | 0.33                                         | Low precision in identifying anomalies.                                                              |
|                                     | Recall (Anomaly -1)                     | 0.14                                         | Low recall, missed many actual anomalies.                                                            |
|                                     | F1-score (Anomaly -1)                   | 0.19                                         | Poor overall performance for the anomaly class.                                                      |
|                                     | Confusion Matrix                        | [[3422, 21615], [6856, 37277]]               | High number of False Positives (21615) and False Negatives (6856) for anomalies.                     |
| **Isolation Forest (Tuned)**        | Best Hyperparameters                    | {'contamination': 0.1, 'n_estimators': 100}               | Hyperparameters found via GridSearchCV.                                                             |
|                                     | Precision (Anomaly -1)                  | 0.4719       | Precision for the anomaly class after tuning.                                                        |
|                                     | Recall (Anomaly -1)                     | 0.1304          | Recall for the anomaly class after tuning.                                                           |
|                                     | F1-score (Anomaly -1)                   | 0.2043              | F1-score for the anomaly class after tuning.                                                         |
|                                     | Confusion Matrix                        | [[3264, 21773], [3653, 40480]]    | Confusion matrix after tuning.                                                                       |
| **Local Outlier Factor (LOF) (Tuned)** | Best Hyperparameters          | {'contamination': 'auto', 'n_neighbors': 50}                            | Hyperparameters found via GridSearchCV.                                                             |
|                                     | Precision (Anomaly -1)                  | 0.3378                    | Precision for the anomaly class after tuning.                                                        |
|                                     | Recall (Anomaly -1)                     | 0.1441                       | Recall for the anomaly class after tuning.                                                           |
|                                     | F1-score (Anomaly -1)                   | 0.2020                           | F1-score for the anomaly class after tuning.                                                         |
|                                     | Confusion Matrix                        | [[3608, 21429], [7074, 37059]]                 | Confusion matrix after tuning.                                                                       |

### Analysis:

*   The untuned Isolation Forest and LOF models showed poor performance in identifying anomalies, particularly in terms of recall.
*   Hyperparameter tuning improved the F1-scores for both Isolation Forest and LOF, but the overall performance in detecting anomalies remains relatively low, especially in terms of recall.
*   The ARI scores for K-Means and DBSCAN indicate limited agreement with the true anomaly labels, suggesting these clustering methods, with their current configurations, are not highly effective for direct anomaly detection in this dataset.
*   The tuned Isolation Forest and LOF models, while showing some improvement, still exhibit a significant number of false positives and false negatives for the anomaly class, highlighting the challenge of this anomaly detection task.

### Conclusion:

Even after hyperparameter tuning, the performance of Isolation Forest and Local Outlier Factor in identifying anomalies is not strong. The low recall scores for the anomaly class across these methods indicate that a large proportion of actual anomalies are being missed. The clustering methods (K-Means and DBSCAN) also did not demonstrate a strong alignment with the true anomaly labels based on ARI. Further efforts, potentially involving more extensive hyperparameter tuning, exploring different anomaly detection algorithms, or feature engineering, are likely required to achieve satisfactory anomaly detection performance on this dataset.


**Reasoning**:
The previous code block failed due to a SyntaxError. I need to fix the syntax error in the markdown formatting and regenerate the summary.



In [35]:
from IPython.display import display, Markdown
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

summary_markdown = """
## Model Performance Comparison Summary

Here's a comparison of the performance metrics for the applied anomaly detection and clustering models, including the results after hyperparameter tuning:

| Model                               | Metric                                  | Score/Value                                  | Notes                                                                                                |
|-------------------------------------|-----------------------------------------|----------------------------------------------|------------------------------------------------------------------------------------------------------|
| **K-Means (Untuned)**               | Adjusted Rand Index (ARI)               | {ari_score:.4f}                              | Measures similarity between clusters and true labels. Moderate agreement.                             |
| **Isolation Forest (Untuned)**      | Precision (Anomaly -1)                  | 0.36                                         | Low precision in identifying anomalies.                                                              |
|                                     | Recall (Anomaly -1)                     | 0.06                                         | Very low recall, missed a large number of actual anomalies.                                          |
|                                     | F1-score (Anomaly -1)                   | 0.10                                         | Poor overall performance for the anomaly class.                                                      |
|                                     | Confusion Matrix                        | [[1441, 23596], [2526, 41607]]               | High number of False Positives (23596) and False Negatives (2526) for anomalies.                     |
| **DBSCAN (Untuned)**                | Adjusted Rand Index (ARI)               | {ari_dbscan:.4f}                             | Very low ARI, poor agreement with true labels.                                                       |
| **Local Outlier Factor (LOF) (Untuned)** | Precision (Anomaly -1)        | 0.33                                         | Low precision in identifying anomalies.                                                              |
|                                     | Recall (Anomaly -1)                     | 0.14                                         | Low recall, missed many actual anomalies.                                                            |
|                                     | F1-score (Anomaly -1)                   | 0.19                                         | Poor overall performance for the anomaly class.                                                      |
|                                     | Confusion Matrix                        | [[3422, 21615], [6856, 37277]]               | High number of False Positives (21615) and False Negatives (6856) for anomalies.                     |
| **Isolation Forest (Tuned)**        | Best Hyperparameters                    | {best_isolation_forest_params}               | Hyperparameters found via GridSearchCV.                                                             |
|                                     | Precision (Anomaly -1)                  | {tuned_isolation_forest_precision:.4f}       | Precision for the anomaly class after tuning.                                                        |
|                                     | Recall (Anomaly -1)                     | {tuned_isolation_forest_recall:.4f}          | Recall for the anomaly class after tuning.                                                           |
|                                     | F1-score (Anomaly -1)                   | {tuned_isolation_forest_f1:.4f}              | F1-score for the anomaly class after tuning.                                                         |
|                                     | Confusion Matrix                        | {tuned_isolation_forest_confusion_matrix}    | Confusion matrix after tuning.                                                                       |
| **Local Outlier Factor (LOF) (Tuned)** | Best Hyperparameters          | {best_lof_params}                            | Hyperparameters found via GridSearchCV.                                                             |
|                                     | Precision (Anomaly -1)                  | {tuned_lof_precision:.4f}                    | Precision for the anomaly class after tuning.                                                        |
|                                     | Recall (Anomaly -1)                     | {tuned_lof_recall:.4f}                       | Recall for the anomaly class after tuning.                                                           |
|                                     | F1-score (Anomaly -1)                   | {tuned_lof_f1:.4f}                           | F1-score for the anomaly class after tuning.                                                         |
|                                     | Confusion Matrix                        | {tuned_lof_confusion_matrix}                 | Confusion matrix after tuning.                                                                       |

### Analysis:

*   The untuned Isolation Forest and LOF models showed poor performance in identifying anomalies, particularly in terms of recall.
*   Hyperparameter tuning improved the F1-scores for both Isolation Forest and LOF, but the overall performance in detecting anomalies remains relatively low, especially in terms of recall.
*   The ARI scores for K-Means and DBSCAN indicate limited agreement with the true anomaly labels, suggesting these clustering methods, with their current configurations, are not highly effective for direct anomaly detection in this dataset.
*   The tuned Isolation Forest and LOF models, while showing some improvement, still exhibit a significant number of false positives and false negatives for the anomaly class, highlighting the challenge of this anomaly detection task.

### Conclusion:

Even after hyperparameter tuning, the performance of Isolation Forest and Local Outlier Factor in identifying anomalies is not strong. The low recall scores for the anomaly class across these methods indicate that a large proportion of actual anomalies are being missed. The clustering methods (K-Means and DBSCAN) also did not demonstrate a strong alignment with the true anomaly labels based on ARI. Further efforts, potentially involving more extensive hyperparameter tuning, exploring different anomaly detection algorithms, or feature engineering, are likely required to achieve satisfactory anomaly detection performance on this dataset.
""".format(
    ari_score=ari_score,
    ari_dbscan=ari_dbscan,
    best_isolation_forest_params=best_isolation_forest_params,
    tuned_isolation_forest_precision=precision_score(binary_true_labels, isolation_forest_predictions, pos_label=-1),
    tuned_isolation_forest_recall=recall_score(binary_true_labels, isolation_forest_predictions, pos_label=-1),
    tuned_isolation_forest_f1=f1_score(binary_true_labels, isolation_forest_predictions, pos_label=-1),
    tuned_isolation_forest_confusion_matrix=confusion_matrix(binary_true_labels, isolation_forest_predictions).tolist(),
    best_lof_params=best_lof_params,
    tuned_lof_precision=precision_score(binary_true_labels, lof_predictions_tuned, pos_label=-1),
    tuned_lof_recall=recall_score(binary_true_labels, lof_predictions_tuned, pos_label=-1),
    tuned_lof_f1=f1_score(binary_true_labels, lof_predictions_tuned, pos_label=-1),
    tuned_lof_confusion_matrix=confusion_matrix(binary_true_labels, lof_predictions_tuned).tolist()
)

display(Markdown(summary_markdown))


## Model Performance Comparison Summary

Here's a comparison of the performance metrics for the applied anomaly detection and clustering models, including the results after hyperparameter tuning:

| Model                               | Metric                                  | Score/Value                                  | Notes                                                                                                |
|-------------------------------------|-----------------------------------------|----------------------------------------------|------------------------------------------------------------------------------------------------------|
| **K-Means (Untuned)**               | Adjusted Rand Index (ARI)               | 0.3299                              | Measures similarity between clusters and true labels. Moderate agreement.                             |
| **Isolation Forest (Untuned)**      | Precision (Anomaly -1)                  | 0.36                                         | Low precision in identifying anomalies.                                                              |
|                                     | Recall (Anomaly -1)                     | 0.06                                         | Very low recall, missed a large number of actual anomalies.                                          |
|                                     | F1-score (Anomaly -1)                   | 0.10                                         | Poor overall performance for the anomaly class.                                                      |
|                                     | Confusion Matrix                        | [[1441, 23596], [2526, 41607]]               | High number of False Positives (23596) and False Negatives (2526) for anomalies.                     |
| **DBSCAN (Untuned)**                | Adjusted Rand Index (ARI)               | 0.0772                             | Very low ARI, poor agreement with true labels.                                                       |
| **Local Outlier Factor (LOF) (Untuned)** | Precision (Anomaly -1)        | 0.33                                         | Low precision in identifying anomalies.                                                              |
|                                     | Recall (Anomaly -1)                     | 0.14                                         | Low recall, missed many actual anomalies.                                                            |
|                                     | F1-score (Anomaly -1)                   | 0.19                                         | Poor overall performance for the anomaly class.                                                      |
|                                     | Confusion Matrix                        | [[3422, 21615], [6856, 37277]]               | High number of False Positives (21615) and False Negatives (6856) for anomalies.                     |
| **Isolation Forest (Tuned)**        | Best Hyperparameters                    | {'contamination': 0.1, 'n_estimators': 100}               | Hyperparameters found via GridSearchCV.                                                             |
|                                     | Precision (Anomaly -1)                  | 0.4719       | Precision for the anomaly class after tuning.                                                        |
|                                     | Recall (Anomaly -1)                     | 0.1304          | Recall for the anomaly class after tuning.                                                           |
|                                     | F1-score (Anomaly -1)                   | 0.2043              | F1-score for the anomaly class after tuning.                                                         |
|                                     | Confusion Matrix                        | [[3264, 21773], [3653, 40480]]    | Confusion matrix after tuning.                                                                       |
| **Local Outlier Factor (LOF) (Tuned)** | Best Hyperparameters          | {'contamination': 'auto', 'n_neighbors': 50}                            | Hyperparameters found via GridSearchCV.                                                             |
|                                     | Precision (Anomaly -1)                  | 0.3378                    | Precision for the anomaly class after tuning.                                                        |
|                                     | Recall (Anomaly -1)                     | 0.1441                       | Recall for the anomaly class after tuning.                                                           |
|                                     | F1-score (Anomaly -1)                   | 0.2020                           | F1-score for the anomaly class after tuning.                                                         |
|                                     | Confusion Matrix                        | [[3608, 21429], [7074, 37059]]                 | Confusion matrix after tuning.                                                                       |

### Analysis:

*   The untuned Isolation Forest and LOF models showed poor performance in identifying anomalies, particularly in terms of recall.
*   Hyperparameter tuning improved the F1-scores for both Isolation Forest and LOF, but the overall performance in detecting anomalies remains relatively low, especially in terms of recall.
*   The ARI scores for K-Means and DBSCAN indicate limited agreement with the true anomaly labels, suggesting these clustering methods, with their current configurations, are not highly effective for direct anomaly detection in this dataset.
*   The tuned Isolation Forest and LOF models, while showing some improvement, still exhibit a significant number of false positives and false negatives for the anomaly class, highlighting the challenge of this anomaly detection task.

### Conclusion:

Even after hyperparameter tuning, the performance of Isolation Forest and Local Outlier Factor in identifying anomalies is not strong. The low recall scores for the anomaly class across these methods indicate that a large proportion of actual anomalies are being missed. The clustering methods (K-Means and DBSCAN) also did not demonstrate a strong alignment with the true anomaly labels based on ARI. Further efforts, potentially involving more extensive hyperparameter tuning, exploring different anomaly detection algorithms, or feature engineering, are likely required to achieve satisfactory anomaly detection performance on this dataset.


**Reasoning**:
Review the previous markdown summary and create a new markdown cell summarizing the findings, reiterating the key results, and explaining why a best model cannot be confidently selected at this stage.



In [36]:
from IPython.display import display, Markdown

summary_text = """
## Summary of Model Comparison and Selection

Based on the evaluation of K-Means, Isolation Forest, DBSCAN, and Local Outlier Factor (LOF) models for anomaly detection on a 10% sample of the dataset, the following key findings were observed:

*   **Untuned Models:** Both Isolation Forest and LOF, intended for anomaly detection, showed poor initial performance with very low recall for the anomaly class, meaning they missed a large proportion of actual anomalies. K-Means and DBSCAN, while clustering the data, did not show a strong alignment with the true anomaly labels based on their Adjusted Rand Index (ARI) scores.
*   **Tuned Models:** Hyperparameter tuning using cross-validation improved the F1-scores for Isolation Forest and LOF, but the recall for the anomaly class remained low. This indicates that even with optimized parameters, these models still struggled to effectively identify anomalies.

**Rationale for Model Selection:**

Given the consistently low performance metrics, particularly the low recall for the anomaly class across all evaluated models (both untuned and tuned), **it is not possible to confidently select a "best" model** at this stage. None of the models demonstrated sufficient capability to effectively distinguish anomalies from normal instances in this dataset sample with their current configurations.

**Next Steps:**

To improve anomaly detection performance on this dataset, further efforts are necessary. These may include:

*   More extensive hyperparameter tuning, potentially exploring a wider range of values or different tuning strategies.
*   Investigating alternative anomaly detection algorithms that might be better suited to the specific characteristics of the dataset.
*   Refining the data preprocessing steps, such as feature engineering or different scaling methods.
*   Analyzing the distribution of anomalies within the dataset to gain further insights into their characteristics.
"""

display(Markdown(summary_text))


## Summary of Model Comparison and Selection

Based on the evaluation of K-Means, Isolation Forest, DBSCAN, and Local Outlier Factor (LOF) models for anomaly detection on a 10% sample of the dataset, the following key findings were observed:

*   **Untuned Models:** Both Isolation Forest and LOF, intended for anomaly detection, showed poor initial performance with very low recall for the anomaly class, meaning they missed a large proportion of actual anomalies. K-Means and DBSCAN, while clustering the data, did not show a strong alignment with the true anomaly labels based on their Adjusted Rand Index (ARI) scores.
*   **Tuned Models:** Hyperparameter tuning using cross-validation improved the F1-scores for Isolation Forest and LOF, but the recall for the anomaly class remained low. This indicates that even with optimized parameters, these models still struggled to effectively identify anomalies.

**Rationale for Model Selection:**

Given the consistently low performance metrics, particularly the low recall for the anomaly class across all evaluated models (both untuned and tuned), **it is not possible to confidently select a "best" model** at this stage. None of the models demonstrated sufficient capability to effectively distinguish anomalies from normal instances in this dataset sample with their current configurations.

**Next Steps:**

To improve anomaly detection performance on this dataset, further efforts are necessary. These may include:

*   More extensive hyperparameter tuning, potentially exploring a wider range of values or different tuning strategies.
*   Investigating alternative anomaly detection algorithms that might be better suited to the specific characteristics of the dataset.
*   Refining the data preprocessing steps, such as feature engineering or different scaling methods.
*   Analyzing the distribution of anomalies within the dataset to gain further insights into their characteristics.
